<div align="center">

# Features Evaluation
</div>

In [1]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path


In [2]:
# -----------------------------
# 1. LOAD DATA
# -----------------------------
file_path = "../Features/Seizure/chb01_15_seizure.mat"

with h5py.File(file_path, 'r') as f:
    X = f['all_feats'][:]          # (windows, 1530)
    y = f['all_labels'][:].squeeze()

print("X shape:", X.shape)
print("y distribution:", np.bincount(y.astype(int)))


X shape: (1437, 1530)
y distribution: [1421   16]


# 1. Load Features

In [3]:
# List all feature files
feature_dir = Path("../Features/Seizure")

files = sorted(feature_dir.glob("chb01_*_seizure.mat"))

print(len(files))
for f in files:
    print(f.name)

7
chb01_03_seizure.mat
chb01_04_seizure.mat
chb01_15_seizure.mat
chb01_16_seizure.mat
chb01_18_seizure.mat
chb01_21_seizure.mat
chb01_26_seizure.mat


In [4]:
# Load records into a dictionary

records = {}

for file in files:

    with h5py.File(file, "r") as f:

        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    # If features are stored as (1530, n_windows)
    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_features", "")

    records[record_name] = (X, y)

    print(record_name, X.shape, y.shape)

chb01_03_seizure (1437, 1530) (1437,)
chb01_04_seizure (1437, 1530) (1437,)
chb01_15_seizure (1437, 1530) (1437,)
chb01_16_seizure (1437, 1530) (1437,)
chb01_18_seizure (1437, 1530) (1437,)
chb01_21_seizure (1437, 1530) (1437,)
chb01_26_seizure (927, 1530) (927,)


In [5]:
# Put inside list
X_all = []
y_all = []

for name, (X, y) in records.items():
    X_all.append(X)
    y_all.append(y)

X_all = np.vstack(X_all)
y_all = np.hstack(y_all)

print(X_all.shape, y_all.shape)

(9549, 1530) (9549,)


## 1.1 Separate in Train/Test

In [6]:
# from sklearn.feature_selection import f_classif
# import numpy as np

# results = {}

# for test_record in records.keys():

#     print("\nTesting:", test_record)

#     # -----------------------
#     # Split LOSO
#     # -----------------------
#     X_train = []
#     y_train = []

#     X_test, y_test = records[test_record]

#     for name, (X, y) in records.items():

#         if name != test_record:
#             X_train.append(X)
#             y_train.append(y)

#     X_train = np.vstack(X_train)
#     y_train = np.hstack(y_train)


#     # -----------------------
#     # Feature selection ONLY on train
#     # -----------------------
#     F, p = f_classif(X_train, y_train)

#     idx = np.argsort(F)[::-1]

#     K = 100   # choose later
#     selected = idx[:K]


#     # Apply same features to test
#     X_train_sel = X_train[:, selected]
#     X_test_sel  = X_test[:, selected]


#     print("Train:", X_train_sel.shape)
#     print("Test :", X_test_sel.shape)

In [7]:
records.keys()

dict_keys(['chb01_03_seizure', 'chb01_04_seizure', 'chb01_15_seizure', 'chb01_16_seizure', 'chb01_18_seizure', 'chb01_21_seizure', 'chb01_26_seizure'])

In [8]:
# Separate 
results = {}

test_record = 'chb01_03_seizure'
# -----------------------
# Split LOSO
# -----------------------
X_train = []
y_train = []

X_test, y_test = records[test_record]

for name, (X, y) in records.items():

    if name != test_record:
        X_train.append(X)
        y_train.append(y)

X_train = np.vstack(X_train)
y_train = np.hstack(y_train)


In [ ]:
import numpy as np
import pandas as pd

# ==================================
# Correlation filter (TRAIN ONLY)
# ==================================

# Feature-feature correlation
corr_matrix = pd.DataFrame(X_train).corr().abs()

print("Correlation matrix:", corr_matrix.shape)
# should print (1530, 1530)


# Keep only upper triangle (avoid duplicate pairs)
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)


# Threshold
threshold = 0.95


# Find features to remove
to_drop = [
    col for col in upper.columns
    if any(upper[col] > threshold)
]


print("Highly correlated features:", len(to_drop))


# ==================================
# Apply same removal to train and test
# ==================================

keep_features = [
    i for i in range(X_train.shape[1])
    if i not in to_drop
]


X_train_corr = X_train[:, keep_features]
X_test_corr  = X_test[:, keep_features]


print("Before:", X_train.shape, X_test.shape)
print("After :", X_train_corr.shape, X_test_corr.shape)

In [ ]:
import numpy as np

from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)


# ============================
# Choose test record (LOSO)
# ============================

test_record = "chb01_03_seizure"

X_test, y_test = records[test_record]

X_train = []
y_train = []

for name, (X, y) in records.items():

    if name != test_record:
        X_train.append(X)
        y_train.append(y)


X_train = np.vstack(X_train)
y_train = np.hstack(y_train)


print("Train:", X_train.shape)
print("Test :", X_test.shape)


# ============================
# Feature selection (TRAIN ONLY)
# ============================

F, p = f_classif(X_train, y_train)

idx = np.argsort(F)[::-1]


K = 50  # test different values later

selected_features = idx[:K]


X_train_sel = X_train[:, selected_features]
X_test_sel  = X_test[:, selected_features]


print("Selected features:", X_train_sel.shape)


# ============================
# Scaling (TRAIN ONLY)
# ============================

scaler = StandardScaler()

X_train_sel = scaler.fit_transform(X_train_sel)
X_test_sel  = scaler.transform(X_test_sel)


# ============================
# Train SVM
# ============================

clf = SVC(
    kernel="rbf",
    probability=True,
    class_weight='balanced',
    C=0.1,
    gamma="scale",
    random_state=42
)

clf.fit(X_train_sel, y_train)


# ============================
# Evaluation
# ============================

y_pred = clf.predict(X_test_sel)
y_prob = clf.predict_proba(X_test_sel)[:, 1]


# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()


accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, 
    y_pred,
    zero_division=0
)

sensitivity = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

specificity = tn / (tn + fp)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)


auc = roc_auc_score(y_test, y_prob)
auprc = average_precision_score(y_test, y_prob)


print(f"Accuracy     : {accuracy:.3f}")
print(f"Precision    : {precision:.3f}")
print(f"Sensitivity  : {sensitivity:.3f}")
print(f"Specificity  : {specificity:.3f}")
print(f"F1-score     : {f1:.3f}")
print(f"ROC-AUC      : {auc:.3f}")
print(f"AUPRC        : {auprc:.3f}")

Train: (8112, 1530)
Test : (1437, 1530)
Selected features: (8112, 100)
Accuracy     : 0.981
Precision    : 0.366
Sensitivity  : 0.938
Specificity  : 0.982
F1-score     : 0.526
ROC-AUC      : 0.992
AUPRC        : 0.541


In [37]:
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)


# =====================================================
# PARAMETERS
# =====================================================

K_VALUES = [20]

param_grid = {
    "select__k": K_VALUES,
    "svc__C": [0.01, 0.1],
    "svc__gamma": ["scale"]
}


inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)


# =====================================================
# LOSO LOOP
# =====================================================

results = {}


for test_record in records.keys():

    print("\n===================================")
    print("Testing:", test_record)
    print("===================================")


    # -----------------------------
    # Split LOSO
    # -----------------------------

    X_test, y_test = records[test_record]


    X_train = []
    y_train = []


    for name, (X, y) in records.items():

        if name != test_record:

            X_train.append(X)
            y_train.append(y)


    X_train = np.vstack(X_train)
    y_train = np.hstack(y_train)



    print("Train:", X_train.shape)
    print("Test :", X_test.shape)



    # -----------------------------
    # Pipeline
    # -----------------------------

    pipe = Pipeline([

        ("select",
         SelectKBest(
             score_func=f_classif
         )),

        ("scale",
         StandardScaler()),

        ("svc",
         SVC(
             kernel="rbf",
             probability=True,
             class_weight="balanced",
             random_state=0
         ))
    ])



    # -----------------------------
    # Inner CV tuning
    # -----------------------------

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="average_precision",
        cv=inner_cv,
        n_jobs=-1,
        refit=True
    )


    search.fit(
        X_train,
        y_train
    )


    print(
        "Best params:",
        search.best_params_
    )

    print(
        "Best inner AUPRC:",
        search.best_score_
    )



    # -----------------------------
    # Test held-out record
    # -----------------------------

    model = search.best_estimator_


    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:,1]



    # -----------------------------
    # Metrics
    # -----------------------------

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )


    sensitivity = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )


    specificity = tn / (tn + fp)


    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )


    auc = roc_auc_score(
        y_test,
        y_prob
    )


    auprc = average_precision_score(
        y_test,
        y_prob
    )



    print(f"Accuracy    : {accuracy:.3f}")
    print(f"Precision   : {precision:.3f}")
    print(f"Sensitivity : {sensitivity:.3f}")
    print(f"Specificity : {specificity:.3f}")
    print(f"F1-score    : {f1:.3f}")
    print(f"ROC-AUC     : {auc:.3f}")
    print(f"AUPRC       : {auprc:.3f}")



    results[test_record] = {

        "best_params": search.best_params_,
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1,
        "auc": auc,
        "auprc": auprc
    }



# =====================================================
# SUMMARY
# =====================================================

print("\n\n===== FINAL LOSO SUMMARY =====")


for metric in [
    "accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1",
    "auc",
    "auprc"
]:

    values = [
        results[r][metric]
        for r in results
    ]

    print(
        f"{metric:12s}: "
        f"{np.mean(values):.3f} ± {np.std(values):.3f}"
    )


Testing: chb01_03_seizure
Train: (8112, 1530)
Test : (1437, 1530)


KeyboardInterrupt: 